# RAG Query Engine and Persistent Chatbot

This notebook is now organized like a small RAG application instead of a pile of helper functions. The same behavior is kept, but the responsibilities are grouped into three modules:

1. `ChromaKnowledgeBase`: reconnects to ChromaDB, rebuilds the LlamaIndex vector index, rebuilds BM25, and creates the hybrid retriever.
2. `JsonChatSessionStore`: saves and loads chat memory as one JSON file per chat in the `session` folder.
3. `RagNewsChatbot`: exposes the product-facing actions: ask one question, open a chat, continue a chat, inspect history, and show sources.

```mermaid
flowchart LR
    U[User question] --> APP[RagNewsChatbot]
    APP --> HR[HybridRetriever]
    HR --> VS[Semantic search\nChroma vector index]
    HR --> BM[Keyword search\nBM25]
    VS --> RRF[Reciprocal rank fusion]
    BM --> RRF
    RRF --> LLM[Gemma 3 1B via Ollama]
    LLM --> A[Answer + sources]
    APP <--> MEM[ChatMemoryBuffer]
    MEM <--> JSON[session/*.json]
```

The important product idea is separation of concerns: retrieval, generation, and persistence are related, but they should not be tangled together.

In [ ]:
# Run this once if the LlamaIndex integrations are missing, then restart the kernel.
%pip install llama-index-vector-stores-chroma llama-index-retrievers-bm25 llama-index-llms-ollama

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "app.py").exists() and (project_root.parent / "app.py").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import importlib
import src.rag.chatbot as rag_chatbot_module
import src.rag.factory as rag_factory_module
import src.rag.observability as rag_observability_module
import src.rag.session_store as rag_session_store_module

importlib.reload(rag_session_store_module)
importlib.reload(rag_observability_module)
importlib.reload(rag_chatbot_module)
importlib.reload(rag_factory_module)

from src.rag import (
    DEFAULT_COLLECTION_NAME,
    DEFAULT_EMBED_MODEL_NAME,
    DEFAULT_HUGGINGFACE_MODEL_KEY,
    DEFAULT_LLM_PROVIDER,
    DEFAULT_OLLAMA_MODEL,
    DEFAULT_PHOENIX_ENDPOINT,
    DEFAULT_PHOENIX_PROJECT_NAME,
    HUGGINGFACE_CHAT_MODELS,
    LLM_PROVIDER_HUGGINGFACE,
    LLM_PROVIDER_OLLAMA,
    PHOENIX_INSTALL_COMMAND,
    ChromaKnowledgeBase,
    JsonChatSessionStore,
    create_rag_app,
    default_paths,
    print_sources,
    print_response,
    setup_phoenix_observability,
    trace_chat_session,
)

resource module not available on Windows


In [2]:
PROJECT_ROOT = project_root
paths = default_paths(PROJECT_ROOT)
NEWS_DIR = paths.news_dir
CHROMA_DIR = paths.chroma_dir
SESSION_DIR = paths.session_dir
COLLECTION_NAME = DEFAULT_COLLECTION_NAME
EMBED_MODEL_NAME = DEFAULT_EMBED_MODEL_NAME
LLM_PROVIDER = DEFAULT_LLM_PROVIDER  # Choose: LLM_PROVIDER_OLLAMA or LLM_PROVIDER_HUGGINGFACE
OLLAMA_MODEL = DEFAULT_OLLAMA_MODEL
HUGGINGFACE_MODEL = DEFAULT_HUGGINGFACE_MODEL_KEY  # deepseek_v3, minimax_m3, qwen_3_5, or any full HF model id
HUGGINGFACE_PROVIDER = "auto"
PHOENIX_PROJECT_NAME = DEFAULT_PHOENIX_PROJECT_NAME
PHOENIX_COLLECTOR_ENDPOINT = DEFAULT_PHOENIX_ENDPOINT
LAUNCH_PHOENIX_SERVER = False  # Recommended: run `phoenix serve` in a terminal, then keep this False.
PHOENIX_SERVER_COMMAND = "phoenix serve"

print(f"Project root: {PROJECT_ROOT}")
print(f"News source folder: {NEWS_DIR}")
print(f"ChromaDB folder: {CHROMA_DIR}")
print(f"Session folder: {SESSION_DIR}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"LLM provider: {LLM_PROVIDER}")
print(f"Ollama model: {OLLAMA_MODEL}")
print(f"Hugging Face model: {HUGGINGFACE_CHAT_MODELS.get(HUGGINGFACE_MODEL, HUGGINGFACE_MODEL)}")
print(f"Phoenix project: {PHOENIX_PROJECT_NAME}")
print(f"Phoenix collector endpoint: {PHOENIX_COLLECTOR_ENDPOINT}")
print(f"Phoenix server command: {PHOENIX_SERVER_COMMAND}")

Project root: C:\Program Files\Studying\coding\RAG_project
News source folder: C:\Program Files\Studying\coding\RAG_project\data\hk_free_press_news
ChromaDB folder: C:\Program Files\Studying\coding\RAG_project\chromadb_store
Session folder: C:\Program Files\Studying\coding\RAG_project\session
Collection name: news_chat
Embedding model: BAAI/bge-small-en-v1.5
LLM provider: ollama
Ollama model: gemma3:1b
Hugging Face model: deepseek-ai/DeepSeek-V3-0324
Phoenix project: rag-news-chatbot
Phoenix collector endpoint: http://localhost:6006
Phoenix server command: phoenix serve


In [ ]:
# Retrieval, fusion, and knowledge-base classes are now centralized in src/rag/.
# This notebook imports and uses the same implementation as app.py.

In [ ]:
# Session persistence and chatbot facade now come from src/rag/.
# This removes duplicate maintenance between the notebook and app.py.

## Build the Application Objects

Run the next cell once per notebook session. It creates:

- `knowledge_base`: reconnects to the persisted `news_chat` ChromaDB collection
- `session_store`: reads and writes chat JSON files under `session/`
- `rag_app`: the single object you use for query, chat, sources, and history

This is the main readability improvement: examples below call methods on `rag_app` instead of manually coordinating many functions and global variables.

## Run the RAG App

The next cells demonstrate the same functionality as before, but through one object: `rag_app`.

Use `rag_app.ask(...)` for one-off questions. Use `rag_app.open_chat(...)` and `rag_app.chat(...)` for multi-turn conversations with persistent memory.

### Initialize the App

In [3]:
# Run this once after reopening the notebook.
# For stability on Windows, start Phoenix in a terminal with `phoenix serve`
# and keep LAUNCH_PHOENIX_SERVER = False in the config cell above.
# Phoenix must be configured before create_rag_app so LlamaIndex query/chat spans are captured.
phoenix_status = setup_phoenix_observability(
    project_name=PHOENIX_PROJECT_NAME,
    endpoint=PHOENIX_COLLECTOR_ENDPOINT,
    launch_server=LAUNCH_PHOENIX_SERVER,
    batch=True,
    raise_on_missing=False,
)
print(phoenix_status.message)
print(f"Phoenix UI: {phoenix_status.ui_url}")
print(f"Phoenix collector endpoint: {phoenix_status.endpoint}")
if not phoenix_status.enabled:
    print(f"Install Phoenix packages first: {PHOENIX_INSTALL_COMMAND}")
    print(f"Then start server in terminal: {PHOENIX_SERVER_COMMAND}")

# It reconnects to ChromaDB, rebuilds semantic + BM25 retrieval, and prepares chat persistence.
rag_app = create_rag_app(
    chroma_dir=CHROMA_DIR,
    session_dir=SESSION_DIR,
    news_dir=NEWS_DIR,
    collection_name=COLLECTION_NAME,
    embed_model_name=EMBED_MODEL_NAME,
    llm_provider=LLM_PROVIDER,
    ollama_model=OLLAMA_MODEL,
    huggingface_model=HUGGINGFACE_MODEL,
    huggingface_provider=HUGGINGFACE_PROVIDER,
    final_top_k=5,
)
knowledge_base = rag_app.knowledge_base

def notebook_chat(chat_id: str, message: str):
    # Attach the saved chat id to every span so Phoenix groups traces by conversation.
    with trace_chat_session(chat_id, metadata={"interface": "notebook"}):
        return rag_app.chat(chat_id, message)

print(f"Stored vector count: {knowledge_base.count()}")
print("RAG app is ready.")

c:\Users\jason\miniconda3\envs\ai\Lib\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


OpenTelemetry Tracing Details
|  Phoenix Project: rag-news-chatbot
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix tracing is enabled. Run RAG queries, then open the Phoenix UI to inspect traces.
Phoenix UI: http://localhost:6006
Phoenix collector endpoint: http://localhost:6006/v1/traces
Stored vector count: 1499
RAG app is ready.


### Single-turn RAG: use this when the question does not need chat history.

In [ ]:
sample_query = "What model you are ? and what is the latest information you can provide to me ?"
query_response = rag_app.ask(sample_query)

print_response(query_response)
print("-" * 70)
print_sources(query_response)

### Multi-turn RAG With Saved Chat History

`rag_app.open_chat(...)` loads the saved JSON file when it exists, or creates a new one when it does not. Every `rag_app.chat(...)` call saves the updated conversation automatically.

In [ ]:
# Create or load one persistent chat session.
# If session/Sanae_Takaichi.json exists, this restores its previous messages.
chat_id = rag_app.open_chat("Sanae Takaichi")

chat_response = notebook_chat(
    chat_id=chat_id,
    message="Who is Sanae Takaichi ?",
)

print_response(chat_response)
print('-' * 70)
print_sources(chat_response)

In [ ]:
# Follow-up question: this uses the same saved chat memory, so "she" refers to Sanae Takaichi.
follow_up_response = notebook_chat(
    chat_id=chat_id,
    message="Is there a possible Japan will have a war with China in the future ?",
)
print_sources(follow_up_response)

In [ ]:
# Inspect the current in-memory history for this chat.
rag_app.show_history(chat_id)

In [ ]:
# List saved chat files on disk.
rag_app.list_saved_chats()

### Reopen an Existing Chat

After reopening the notebook, run the import/config/module cells and the app initialization cell first. Then `rag_app.open_chat("Sanae Takaichi")` restores the saved conversation from `session/Sanae_Takaichi.json`.

In [4]:
loaded_chat_id = rag_app.open_chat("Sanae Takaichi")

In [5]:
rag_app.show_history(loaded_chat_id)

user: Who is Sanae Takaichi ?
assistant: According to the provided documents, Sanae Takaichi is a Japanese Prime Minister. She will host South Korea’s President Lee Jae Myung for talks on Tuesday aimed at demonstrating their cordial ties as China flexes muscles.
----------------------------------------
user: How long did she talk with Donald Trump during a phone call?
assistant: During a phone call, Sanae Takaichi and Donald Trump spoke for 25 minutes.
----------------------------------------
user: Is there a possible Japan will have a war with China in the future ?
assistant: The documents do not explicitly state whether Japan will have a war with China in the future. However, the documents suggest a heightened state of tension and a cautious approach to the situation, raising concerns about potential conflict.
----------------------------------------


In [6]:
# Continue a loaded chat without rerunning the original conversation cells.
reloaded_response = notebook_chat(
    chat_id=loaded_chat_id,
    message="What is the relationship between Japan and south Korea ?",
)

print_response(reloaded_response)
print('-' * 70)
print_sources(reloaded_response)

According to the provided documents, the relationship between Japan and South Korea is characterized as “bitter memories” stemming from Japan’s brutal occupation of the Korean peninsula from 1910 to 1945. The documents highlight a long history of strained relations, marked by mistrust and a complex interplay of historical grievances and current geopolitical dynamics.
----------------------------------------------------------------------

Source 1: 13-01-2026 | Japan, South Korea leaders meet as China flexes muscles
Lee and Takaichi are also expected to discuss their relations with the United States because the unpredictable Trump “has put in doubt old certainties and highlighted the importance of strengthening their ties”, he said.

Yee Kuang Heng, a professor in international security at the University of Tokyo, did not expect Lee to bring any particular message from Xi to Takaichi.

“However, the two leaders may discuss the fallout from China’s economic coercion that both ROK (South 

### Session Management Helpers

Use these APIs to inspect and manage persisted chat sessions:
- `rag_app.count_chat_ids()`
- `rag_app.list_chat_ids()`
- `rag_app.rename_chat(old_chat_id, new_chat_id)`
- `rag_app.delete_chat(chat_id)`

In [ ]:
# Count and list chat ids currently saved on disk.
print("Saved chat count:", rag_app.count_chat_ids())
print("Saved chat ids:", rag_app.list_chat_ids())

In [ ]:
for delete_chat_id in ['Sanae_Takaichi']:
    rag_app.delete_chat(delete_chat_id, close_open_session=True, missing_ok=False)